In [ ]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine, text


# database connections 


In [ ]:
with open('../../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['OLTP']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)


# Extract


In [ ]:
dim_estado = pd.read_sql_table('mensajeria_estado', co_sa)
dim_estado


# Transformations


In [ ]:
dim_estado.rename(columns={'id': 'id_estado'}, inplace=True)
dim_estado.replace({np.nan: 'NO APLICA', ' ': 'NO APLICA', '': 'NO APLICA'}, inplace=True)
unknown = pd.DataFrame([{'id_estado': -1, 'nombre': 'NO APLICA', 'descripcion': 'NO APLICA'}])
dim_estado = pd.concat([unknown, dim_estado], ignore_index=True)
dim_estado["saved"] = date.today()
dim_estado


# load


In [ ]:
with etl_conn.begin() as conn:
    conn.execute(text('Delete from dim_estado'))
dim_estado.to_sql('dim_estado', etl_conn, if_exists='append', index=False)
